# Experiments – Edouard

Each experiment is a DLC shuffle that holds out one session (train on the 7 others).
Needs `dlc/pupil-dani-2026-11-13/` with its `labeled-data/`. Steps already done are skipped, so "Run All" is safe.

In [2]:
from pathlib import Path

import deeplabcut
import pandas as pd
from deeplabcut.utils import auxiliaryfunctions

PROJECT = Path("dlc") / "pupil-dani-2026-11-13"
MODELS_DIR = PROJECT / "dlc-models-pytorch" / "iteration-0"
EVAL_DIR = PROJECT / "evaluation-results-pytorch" / "iteration-0"
config_path = str(PROJECT / "config.yaml")

cfg = auxiliaryfunctions.read_config(config_path)
cfg["project_path"] = str(PROJECT.resolve())
auxiliaryfunctions.write_config(config_path, cfg)

merged = pd.read_hdf(next((PROJECT / auxiliaryfunctions.get_training_set_folder(cfg)).glob("CollectedData_*.h5")))
frame_sessions = [str(idx[1]) for idx in merged.index]
print(pd.Series(frame_sessions).value_counts().sort_index().to_string())  # 393 labeled frames over 8 sessions


def split_indices(test_session: str) -> tuple[list[int], list[int]]:
    """Return the (train, test) row indices of the merged labels, holding out one session."""
    test = [i for i, session in enumerate(frame_sessions) if session == test_session]
    train = [i for i, session in enumerate(frame_sessions) if session != test_session]
    if not test:
        raise ValueError(f"No labeled frames for session {test_session}.")
    return train, test


def shuffle_dir(shuffle: int) -> Path | None:
    """Return a shuffle's model folder, or None if the shuffle doesn't exist yet."""
    matches = list(MODELS_DIR.glob(f"*shuffle{shuffle}"))
    return matches[0] if matches else None


def create_shuffle(shuffle: int, test_session: str) -> None:
    """Create a shuffle holding out one session, unless the shuffle already exists."""
    if shuffle_dir(shuffle) is not None:
        print(f"Shuffle {shuffle} already exists, skipped.")
        return
    train, test = split_indices(test_session)
    print(f"Shuffle {shuffle}: test on {test_session} | train frames: {len(train)} | test frames: {len(test)}")
    # DLC names a shuffle after its train fraction and requires that fraction in config.yaml's TrainingFraction list.
    fraction = round(len(train) / (len(train) + len(test)), 2)
    project_cfg = auxiliaryfunctions.read_config(config_path)
    if fraction not in project_cfg["TrainingFraction"]:
        project_cfg["TrainingFraction"].append(fraction)
        auxiliaryfunctions.write_config(config_path, project_cfg)
    deeplabcut.create_training_dataset(
        config_path, Shuffles=[shuffle],
        trainIndices=[train], testIndices=[test],
        net_type="resnet_50",
    )


def trainingset_index(shuffle: int) -> int:
    """Position of a shuffle's train fraction in config.yaml's TrainingFraction list, as DLC's functions expect."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    percent = int(folder.name.split("trainset")[1].split("shuffle")[0])
    fractions = auxiliaryfunctions.read_config(config_path)["TrainingFraction"]
    return [round(100 * f) for f in fractions].index(percent)


def configure_shuffle(shuffle: int, train: dict, inference: dict, epochs: int) -> None:
    """Overwrite chosen training settings in a shuffle's pytorch_config.yaml."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    path = folder / "train" / "pytorch_config.yaml"
    pcfg = auxiliaryfunctions.read_plainconfig(path)
    pcfg["data"]["train"].update(train)
    pcfg["data"]["inference"].update(inference)
    pcfg["train_settings"]["epochs"] = epochs
    auxiliaryfunctions.write_plainconfig(path, pcfg)
    print(f"Shuffle {shuffle} | train: {train} | inference: {inference} | epochs: {epochs}")


def train_shuffle(shuffle: int, epochs: int) -> None:
    """Train a shuffle, unless its final snapshot already exists."""
    folder = shuffle_dir(shuffle)
    assert folder is not None, f"Shuffle {shuffle} doesn't exist."
    if (folder / "train" / f"snapshot-{epochs:03d}.pt").exists():
        print(f"Shuffle {shuffle} already trained, skipped.")
        return
    deeplabcut.train_network(config_path, shuffle=shuffle, trainingsetindex=trainingset_index(shuffle))


def evaluate_shuffle(shuffle: int, epochs: int, force: bool = False) -> None:
    """Evaluate a shuffle's final snapshot, unless already done (never the 'best' one, picked with the test score).

    hist_eq is random at test time in DLC, so re-evaluating can change the numbers: use force=True on purpose only.
    """
    done = list(EVAL_DIR.glob(f"*shuffle{shuffle}/*_snapshot_{epochs:03d}-results.csv"))
    if done and not force:
        print(f"Shuffle {shuffle} already evaluated, skipped.")
        return
    deeplabcut.evaluate_network(
        config_path, Shuffles=[shuffle], trainingsetindex=trainingset_index(shuffle),
        snapshots_to_evaluate=[f"snapshot-{epochs:03d}"],
        per_keypoint_evaluation=True,
    )

094124    40
110535    39
115300    34
131707    60
135059    40
171101    20
172731    80
214411    80


## Shuffle 2 – histogram equalization

`hist_eq` in train and inference (DLC applies it to each image with probability 0.5), 100 epochs.
Result on 110535: 93 px mean error (baseline 238 px), 7 % detection (baseline 0 %).

In [4]:
create_shuffle(2, test_session="110535")
configure_shuffle(2, train={"hist_eq": True}, inference={"hist_eq": True}, epochs=100)
train_shuffle(2, epochs=100)
evaluate_shuffle(2, epochs=100)

Shuffle 2 already exists, skipped.
Shuffle 2 | train: {'hist_eq': True} | inference: {'hist_eq': True} | epochs: 100
Shuffle 2 already trained, skipped.
Shuffle 2 already evaluated, skipped.


## Shuffle 3 – histogram equalization + horizontal flip

Same as shuffle 2, plus `hflip: 0.5` in training only (no keypoint swap: labels are anatomical).
Result on 110535: 21 px mean error, 36 % detection; the anterior/posterior swap is gone.

In [5]:
create_shuffle(3, test_session="110535")
configure_shuffle(3, train={"hist_eq": True, "hflip": 0.5}, inference={"hist_eq": True}, epochs=100)
train_shuffle(3, epochs=100)
evaluate_shuffle(3, epochs=100)

Shuffle 3 already exists, skipped.
Shuffle 3 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 3 already trained, skipped.
Shuffle 3 already evaluated, skipped.


## Development rounds – 135059 and 171101 held out

Does shuffle 3's gain hold on other sessions? Each round compares DLC's defaults with `hist_eq` + flip, both at 100 epochs.

| Shuffle | Held out | Settings |
|---|---|---|
| 4 | 135059 | DLC defaults |
| 5 | 135059 | hist_eq + hflip |
| 6 | 171101 | DLC defaults |
| 7 | 171101 | hist_eq + hflip |

About 2 h 30 of training in total.

In [6]:
BASELINE: dict[str, dict] = {"train": {}, "inference": {}}
HISTEQ_FLIP: dict[str, dict] = {"train": {"hist_eq": True, "hflip": 0.5}, "inference": {"hist_eq": True}}

DEV_RUNS: dict[int, tuple[str, dict[str, dict]]] = {
    4: ("135059", BASELINE),
    5: ("135059", HISTEQ_FLIP),
    6: ("171101", BASELINE),
    7: ("171101", HISTEQ_FLIP),
}

for shuffle, (test_session, settings) in DEV_RUNS.items():
    create_shuffle(shuffle, test_session=test_session)
    configure_shuffle(shuffle, train=settings["train"], inference=settings["inference"], epochs=100)

Shuffle 4 already exists, skipped.
Shuffle 4 | train: {} | inference: {} | epochs: 100
Shuffle 5 already exists, skipped.
Shuffle 5 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 6 already exists, skipped.
Shuffle 6 | train: {} | inference: {} | epochs: 100
Shuffle 7 already exists, skipped.
Shuffle 7 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100


In [7]:
for shuffle in DEV_RUNS:
    train_shuffle(shuffle, epochs=100)

Shuffle 4 already trained, skipped.
Shuffle 5 already trained, skipped.
Shuffle 6 already trained, skipped.
Shuffle 7 already trained, skipped.


In [8]:
for shuffle in DEV_RUNS:
    evaluate_shuffle(shuffle, epochs=100)

Shuffle 4 already evaluated, skipped.
Shuffle 5 already evaluated, skipped.
Shuffle 6 already evaluated, skipped.
Shuffle 7 already evaluated, skipped.


## Shuffles 11–17 – same experiments, fixed 135059 labels

135059 had anterior/posterior swapped in its labels (see NOTES.md), so every run with 135059 in training learned from wrong labels. Labels are fixed now, so redoing all of them, all at 100 epochs (incl. the baseline, sh1 had 200).

| Shuffle | Held out | Settings |
|---|---|---|
| 11 | 110535 | DLC defaults |
| 12 | 110535 | hist_eq |
| 13 | 110535 | hist_eq + hflip |
| 14 | 135059 | DLC defaults |
| 15 | 135059 | hist_eq + hflip |
| 16 | 171101 | DLC defaults |
| 17 | 171101 | hist_eq + hflip |

About 4 h of training in total.

In [9]:
BASELINE: dict[str, dict] = {"train": {}, "inference": {}}
HISTEQ: dict[str, dict] = {"train": {"hist_eq": True}, "inference": {"hist_eq": True}}
HISTEQ_FLIP: dict[str, dict] = {"train": {"hist_eq": True, "hflip": 0.5}, "inference": {"hist_eq": True}}

FIXED_LABEL_RUNS: dict[int, tuple[str, dict[str, dict]]] = {
    11: ("110535", BASELINE),
    12: ("110535", HISTEQ),
    13: ("110535", HISTEQ_FLIP),
    14: ("135059", BASELINE),
    15: ("135059", HISTEQ_FLIP),
    16: ("171101", BASELINE),
    17: ("171101", HISTEQ_FLIP),
}

for shuffle, (test_session, settings) in FIXED_LABEL_RUNS.items():
    create_shuffle(shuffle, test_session=test_session)
    configure_shuffle(shuffle, train=settings["train"], inference=settings["inference"], epochs=100)

Shuffle 11: test on 110535 | train frames: 354 | test frames: 39
You passed a split with the following fraction: 90%
Shuffle 11 | train: {} | inference: {} | epochs: 100
Shuffle 12: test on 110535 | train frames: 354 | test frames: 39
You passed a split with the following fraction: 90%
Shuffle 12 | train: {'hist_eq': True} | inference: {'hist_eq': True} | epochs: 100
Shuffle 13: test on 110535 | train frames: 354 | test frames: 39
You passed a split with the following fraction: 90%
Shuffle 13 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 14: test on 135059 | train frames: 353 | test frames: 40
You passed a split with the following fraction: 90%
Shuffle 14 | train: {} | inference: {} | epochs: 100
Shuffle 15: test on 135059 | train frames: 353 | test frames: 40
You passed a split with the following fraction: 90%
Shuffle 15 | train: {'hist_eq': True, 'hflip': 0.5} | inference: {'hist_eq': True} | epochs: 100
Shuffle 16: test on 171101 | tra

In [10]:
for shuffle in FIXED_LABEL_RUNS:
    train_shuffle(shuffle, epochs=100)

Training with configuration:
config_version: 0
model: backbone={'type': 'ResNet', 'model_name': 'resnet50_gn', 'output_stride': 16, 'freeze_bn_stats': False, 'freeze_bn_weights': False} heads={'bodypart': {'type': 'HeatmapHead', 'weight_init': 'normal', 'predictor': {'type': 'HeatmapPredictor', 'apply_sigmoid': False, 'clip_scores': True, 'location_refinement': True, 'locref_std': 7.2801}, 'target_generator': {'type': 'HeatmapGaussianGenerator', 'num_heatmaps': 8, 'pos_dist_thresh': 17, 'heatmap_mode': 'KEYPOINT', 'gradient_masking': False, 'generate_locref': True, 'locref_std': 7.2801}, 'criterion': {'heatmap': {'type': 'WeightedMSECriterion', 'weight': 1.0}, 'locref': {'type': 'WeightedHuberCriterion', 'weight': 0.05}}, 'heatmap_config': {'channels': [2048, 8], 'kernel_size': [3], 'strides': [2]}, 'locref_config': {'channels': [2048, 16], 'kernel_size': [3], 'strides': [2]}}} backbone_output_channels=2048 neck=None pose_model=None
net_type: resnet_50
method: bu
device: auto
metadata:

In [11]:
for shuffle in FIXED_LABEL_RUNS:
    evaluate_shuffle(shuffle, epochs=100)

Evaluation scorer: DLC_Resnet50_pupilNov13shuffle11_snapshot_100


100%|██████████| 39/39 [00:01<00:00, 22.25it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle11_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle11_snapshot_100-results.csv (pcutoff: 0.6):
train rmse              2.65
train rmse_pcutoff      2.65
train mAP              98.72
train mAR              99.32
test rmse             220.11
test rmse_pcutoff      62.76
test mAP                1.15
test mAR                2.56
Name: (0.9, 11, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle11_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.45px
    eye-posterior:     2.73px
    eye-dorsal:        2.99px
    eye-ventral:       3.19px
    pupil-anterior:    2.19px
    pupil-posterior:   2.35px
    pupil-dorsal:      2.23px
    pupil-ventral:     2.03px
  Test error (px)
    eye-anterior:      360.01px
    eye-posterior:     416.53px
    eye-dorsal:        137.81px
    eye-ventral:       207.15px
    pupil-anterior:    196.27px
    pup

100%|██████████| 39/39 [00:01<00:00, 22.19it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle12_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle12_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             2.85
train rmse_pcutoff     2.85
train mAP             98.23
train mAR             98.93
test rmse             67.85
test rmse_pcutoff      3.50
test mAP              16.61
test mAR              21.79
Name: (0.9, 12, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle12_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.57px
    eye-posterior:     2.98px
    eye-dorsal:        3.49px
    eye-ventral:       3.78px
    pupil-anterior:    2.24px
    pupil-posterior:   2.45px
    pupil-dorsal:      2.17px
    pupil-ventral:     2.14px
  Test error (px)
    eye-anterior:      148.33px
    eye-posterior:     141.95px
    eye-dorsal:        31.88px
    eye-ventral:       40.04px
    pupil-anterior:    55.51px
    pupil-posterio

c:\Users\edoua\miniconda3\envs\ml4science\lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:170: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)
100%|██████████| 39/39 [00:01<00:00, 22.27it/s]


... (4 outputs left out)

Evaluation results file: DLC_Resnet50_pupilNov13shuffle15_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle15_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             3.08
train rmse_pcutoff     3.04
train mAP             97.68
train mAR             98.50
test rmse              7.18
test rmse_pcutoff      7.05
test mAP              93.69
test mAR              95.25
Name: (0.9, 15, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle15_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.74px
    eye-posterior:     3.24px
    eye-dorsal:        3.88px
    eye-ventral:       3.92px
    pupil-anterior:    2.42px
    pupil-posterior:   2.63px
    pupil-dorsal:      2.40px
    pupil-ventral:     2.39px
  Test error (px)
    eye-anterior:      13.62px
    eye-posterior:     13.66px
    eye-dorsal:        9.58px
    eye-ventral:       7.19px
    pupil-anterior:    4.19px
    pupil-posterior:   

100%|██████████| 20/20 [00:00<00:00, 24.31it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle16_snapshot_100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle16_snapshot_100-results.csv (pcutoff: 0.6):
train rmse             2.72
train rmse_pcutoff     2.72
train mAP             98.66
train mAR             99.28
test rmse             32.28
test rmse_pcutoff     16.26
test mAP              17.82
test mAR              22.50
Name: (0.95, 16, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle16_snapshot_100-keypoint-results):
  Train error (px)
    eye-anterior:      3.66px
    eye-posterior:     2.66px
    eye-dorsal:        3.15px
    eye-ventral:       3.37px
    pupil-anterior:    2.23px
    pupil-posterior:   2.42px
    pupil-dorsal:      2.12px
    pupil-ventral:     2.12px
  Test error (px)
    eye-anterior:      132.44px
    eye-posterior:     39.06px
    eye-dorsal:        8.51px
    eye-ventral:       2.23px
    pupil-anterior:    36.52px
    pupil-posterior:

c:\Users\edoua\miniconda3\envs\ml4science\lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:170: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)


In [13]:
# sh17: last epoch was also the best one, so DLC saved the final model as snapshot-best-100
deeplabcut.evaluate_network(
    config_path, Shuffles=[17], trainingsetindex=trainingset_index(17),
    snapshots_to_evaluate=["snapshot-best-100"],
    per_keypoint_evaluation=True,
)

c:\Users\edoua\miniconda3\envs\ml4science\lib\site-packages\deeplabcut\pose_estimation_pytorch\config\data.py:170: UserWarning: Be careful! Do not train pose models with horizontal flips if you have symmetric keypoints!
  build_transforms(v)


Evaluation scorer: DLC_Resnet50_pupilNov13shuffle17_snapshot_best-100


100%|██████████| 20/20 [00:00<00:00, 24.19it/s]


Evaluation results file: DLC_Resnet50_pupilNov13shuffle17_snapshot_best-100-results.csv
Evaluation results for DLC_Resnet50_pupilNov13shuffle17_snapshot_best-100-results.csv (pcutoff: 0.6):
train rmse             2.99
train rmse_pcutoff     2.99
train mAP             97.94
train mAR             98.74
test rmse              3.52
test rmse_pcutoff      3.36
test mAP              98.16
test mAR              98.50
Name: (0.95, 17, 100, -1, 0.6), dtype: float64
Per-bodypart evaluation results (DLC_Resnet50_pupilNov13shuffle17_snapshot_best-100-keypoint-results):
  Train error (px)
    eye-anterior:      3.80px
    eye-posterior:     3.05px
    eye-dorsal:        3.74px
    eye-ventral:       3.85px
    pupil-anterior:    2.33px
    pupil-posterior:   2.46px
    pupil-dorsal:      2.51px
    pupil-ventral:     2.18px
  Test error (px)
    eye-anterior:      4.12px
    eye-posterior:     5.58px
    eye-dorsal:        4.50px
    eye-ventral:       2.57px
    pupil-anterior:    2.34px
    pupil

## Comparison of all experiments

Scores computed by `src/metrics.py` from the predictions DLC saved during evaluation (no retraining).

In [3]:
from src.data import load_labels
from src.metrics import compare_runs, evaluate_run, find_predictions, load_predictions

labels = load_labels(Path("data") / "labeled-data")

EXPERIMENTS = {  # name: (shuffle, snapshot, held-out session)
    "sh1 baseline (200 ep)": (1, 200, "110535"),
    "sh2 hist_eq": (2, 100, "110535"),
    "sh3 hist_eq+flip": (3, 100, "110535"),
    "sh4 baseline": (4, 100, "135059"),
    "sh5 hist_eq+flip": (5, 100, "135059"),
    "sh6 baseline": (6, 100, "171101"),
    "sh7 hist_eq+flip": (7, 100, "171101"),
    # same runs with fixed 135059 labels
    "sh11 baseline": (11, 100, "110535"),
    "sh12 hist_eq": (12, 100, "110535"),
    "sh13 hist_eq+flip": (13, 100, "110535"),
    "sh14 baseline": (14, 100, "135059"),
    "sh15 hist_eq+flip": (15, 100, "135059"),
    "sh16 baseline": (16, 100, "171101"),
    "sh17 hist_eq+flip": (17, 100, "171101"),
}

runs = {}
for name, (shuffle, snapshot, test_session) in EXPERIMENTS.items():
    predictions = load_predictions(find_predictions(PROJECT, shuffle, snapshot))
    runs[name] = evaluate_run(predictions, labels, [test_session])

comparison = compare_runs(runs)
comparison.round(3)

C:\Users\edoua\AppData\Local\Temp\ipykernel_13364\3762272641.py:4: UserWarning: Session 150130 has no CollectedData_dani.h5, skipped.
  labels = load_labels(Path("data") / "labeled-data")


,test_sessions,keypoints,mean_error_px,median_error_px,mean_error_norm,pck,detection_rate,detected_error_px,confident_wrong_rate,false_detection_rate,orientation_accuracy
sh1 baseline (200 ep),110535,312,238.405,167.331,1.571,0.189,0.000,NaN,NaN,NaN,0.231
sh2 hist_eq,110535,312,86.025,24.112,0.567,0.353,0.080,21.909,0.200,NaN,0.051
sh3 hist_eq+flip,110535,312,21.109,12.530,0.139,0.615,0.359,9.539,0.000,NaN,0.974
sh4 baseline,135059,320,8.093,4.848,0.052,0.791,0.894,7.464,0.000,NaN,1.000
sh5 hist_eq+flip,135059,320,7.137,5.060,0.046,0.894,0.975,6.977,0.000,NaN,1.000
sh6 baseline,171101,160,15.260,4.151,0.154,0.881,0.588,7.464,0.032,NaN,0.900
sh7 hist_eq+flip,171101,160,9.559,3.429,0.097,0.925,0.712,4.749,0.000,NaN,1.000
sh11 baseline,110535,312,220.113,97.834,1.450,0.173,0.003,62.764,1.000,NaN,0.231
sh12 hist_eq,110535,312,67.849,17.338,0.447,0.449,0.128,3.496,0.000,NaN,0.154
sh13 hist_eq+flip,110535,312,20.302,11.799,0.134,0.625,0.449,10.469,0.007,NaN,0.974
